# Notebook 17: Diagnostic Analysis of RFF + SH Failure

## Goal
Understand WHY RFF catastrophically failed with SH features (-7.96%) in Notebook 16.

## Hypotheses from CRITICAL_ANALYSIS_NB16.md
1. **Frequency Interference** (most likely): SH and RFF use incompatible frequency representations
2. **Dimensionality Curse**: 100D input × 256 neurons × 25 frequencies = overparameterized
3. **Input Statistics Mismatch**: RFF designed for normalized inputs, SH features have different distribution
4. **Optimization Difficulty**: Complex loss landscape with interdependent parameters

## Experiments
1. **Input normalization**: Standardize SH features before RFF
2. **Training dynamics**: Visualize loss curves, gradient norms
3. **Activation visualization**: Plot learned RFF shapes
4. **Feature analysis**: Examine SH feature statistics
5. **Frequency adaptation**: Try learnable frequencies
6. **Longer training**: 500 epochs instead of 100

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm import tqdm

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from satclip.models.activation_models import (
    build_mlp_with_activation,
    RFFActivation,
    SplineActivation
)
from satclip.datasets import GPWDataModule
from satclip.baselines import get_sh_embedding

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. Load Data and Setup

In [ ]:
# Load GPW data at 15-min resolution
data_module = GPWDataModule(
    data_dir="../data/gpw",
    resolution='15min',
    batch_size=256,
    num_workers=4,
    spatial_blocking=True,
    block_size_deg=5.0,
    seed=42
)
data_module.prepare_data()
data_module.setup('fit')

train_loader = data_module.train_dataloader()
test_loader = data_module.test_dataloader()

# Get one batch to check shapes
batch = next(iter(train_loader))
coords, targets = batch['coords'], batch['target']
print(f"Coords shape: {coords.shape}")
print(f"Targets shape: {targets.shape}")
print(f"Train size: {len(train_loader.dataset)}")
print(f"Test size: {len(test_loader.dataset)}")

## 2. Analyze SH Feature Statistics

In [ ]:
# Generate SH features for training data
L = 10
all_coords = []
all_targets = []
for batch in train_loader:
    all_coords.append(batch['coords'])
    all_targets.append(batch['target'])

all_coords = torch.cat(all_coords, dim=0)
all_targets = torch.cat(all_targets, dim=0)

# Get SH features
sh_features = get_sh_embedding(all_coords.numpy(), L=L)
sh_features_tensor = torch.from_numpy(sh_features).float()

print(f"SH features shape: {sh_features.shape}")
print(f"\nSH Feature Statistics:")
print(f"  Mean: {sh_features.mean(axis=0)[:5]} ... (first 5)")
print(f"  Std:  {sh_features.std(axis=0)[:5]} ... (first 5)")
print(f"  Min:  {sh_features.min(axis=0)[:5]} ... (first 5)")
print(f"  Max:  {sh_features.max(axis=0)[:5]} ... (first 5)")
print(f"\nOverall:")
print(f"  Global mean: {sh_features.mean():.4f}")
print(f"  Global std: {sh_features.std():.4f}")
print(f"  Global min: {sh_features.min():.4f}")
print(f"  Global max: {sh_features.max():.4f}")

In [ ]:
# Visualize SH feature distributions
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('SH Feature Distributions (L=10, 100 features)', fontsize=14)

# Plot histograms for first 6 features
for idx, ax in enumerate(axes.flat):
    ax.hist(sh_features[:, idx], bins=50, alpha=0.7, edgecolor='black')
    ax.set_title(f'Feature {idx}')
    ax.set_xlabel('Value')
    ax.set_ylabel('Frequency')
    ax.axvline(sh_features[:, idx].mean(), color='red', linestyle='--', label='Mean')
    ax.legend()

plt.tight_layout()
plt.savefig('sh_feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nKey observations:")
print(f"- Features are NOT normalized (not zero-mean, unit-variance)")
print(f"- Features have different scales across dimensions")
print(f"- This violates RFF's assumption of normalized inputs!")

## 3. Experiment 1: Input Normalization

Test if standardizing SH features fixes RFF's failure.

In [ ]:
def train_model(model, train_loader, test_loader, epochs=100, lr=1e-3, 
                use_sh=False, L=10, normalize_sh=False, verbose=True):
    """Train a model and track metrics."""
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()
    
    # Compute normalization statistics if needed
    sh_mean, sh_std = None, None
    if use_sh and normalize_sh:
        all_sh = []
        for batch in train_loader:
            coords = batch['coords'].numpy()
            sh = get_sh_embedding(coords, L=L)
            all_sh.append(sh)
        all_sh = np.concatenate(all_sh, axis=0)
        sh_mean = all_sh.mean(axis=0)
        sh_std = all_sh.std(axis=0) + 1e-8  # avoid division by zero
        print(f"Computed SH normalization: mean={sh_mean[:3]}, std={sh_std[:3]}")
    
    train_losses = []
    test_losses = []
    test_r2s = []
    grad_norms = []
    
    for epoch in range(epochs):
        # Training
        model.train()
        epoch_loss = 0.0
        epoch_grad_norm = 0.0
        
        for batch in train_loader:
            coords = batch['coords']
            targets = batch['target'].to(device)
            
            # Get input features
            if use_sh:
                inputs = get_sh_embedding(coords.numpy(), L=L)
                if normalize_sh:
                    inputs = (inputs - sh_mean) / sh_std
                inputs = torch.from_numpy(inputs).float().to(device)
            else:
                inputs = coords.to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs).squeeze()
            loss = criterion(outputs, targets)
            loss.backward()
            
            # Track gradient norm
            total_norm = 0.0
            for p in model.parameters():
                if p.grad is not None:
                    total_norm += p.grad.data.norm(2).item() ** 2
            total_norm = total_norm ** 0.5
            epoch_grad_norm += total_norm
            
            optimizer.step()
            epoch_loss += loss.item()
        
        train_losses.append(epoch_loss / len(train_loader))
        grad_norms.append(epoch_grad_norm / len(train_loader))
        
        # Evaluation
        if (epoch + 1) % 10 == 0 or epoch == 0:
            model.eval()
            test_loss = 0.0
            all_preds = []
            all_targets = []
            
            with torch.no_grad():
                for batch in test_loader:
                    coords = batch['coords']
                    targets = batch['target'].to(device)
                    
                    if use_sh:
                        inputs = get_sh_embedding(coords.numpy(), L=L)
                        if normalize_sh:
                            inputs = (inputs - sh_mean) / sh_std
                        inputs = torch.from_numpy(inputs).float().to(device)
                    else:
                        inputs = coords.to(device)
                    
                    outputs = model(inputs).squeeze()
                    loss = criterion(outputs, targets)
                    test_loss += loss.item()
                    
                    all_preds.append(outputs.cpu())
                    all_targets.append(targets.cpu())
            
            test_losses.append(test_loss / len(test_loader))
            
            # Compute R²
            preds = torch.cat(all_preds)
            targets = torch.cat(all_targets)
            ss_res = ((targets - preds) ** 2).sum()
            ss_tot = ((targets - targets.mean()) ** 2).sum()
            r2 = 1 - ss_res / ss_tot
            test_r2s.append(r2.item())
            
            if verbose:
                print(f"Epoch {epoch+1}/{epochs}: "
                      f"Train Loss={train_losses[-1]:.6f}, "
                      f"Test Loss={test_losses[-1]:.6f}, "
                      f"R²={r2:.4f}, "
                      f"Grad Norm={grad_norms[-1]:.4f}")
    
    return {
        'train_losses': train_losses,
        'test_losses': test_losses,
        'test_r2s': test_r2s,
        'grad_norms': grad_norms,
        'final_r2': test_r2s[-1]
    }

In [ ]:
# Test 1: SH + RFF without normalization (reproduce failure)
print("=" * 60)
print("Test 1: SH + RFF (no normalization) - Should fail")
print("=" * 60)

model_sh_rff_no_norm = build_mlp_with_activation(
    input_dim=100,
    hidden_dims=[256, 256, 256],
    output_dim=256,
    activation_type='rff',
    activation_kwargs={'n_features': 25, 'max_freq': 10.0}
)

results_no_norm = train_model(
    model_sh_rff_no_norm,
    train_loader,
    test_loader,
    epochs=100,
    use_sh=True,
    L=10,
    normalize_sh=False
)

print(f"\nFinal R² (no normalization): {results_no_norm['final_r2']:.4f}")

In [ ]:
# Test 2: SH + RFF with normalization (should help?)
print("\n" + "=" * 60)
print("Test 2: SH + RFF (WITH normalization) - Will this fix it?")
print("=" * 60)

model_sh_rff_norm = build_mlp_with_activation(
    input_dim=100,
    hidden_dims=[256, 256, 256],
    output_dim=256,
    activation_type='rff',
    activation_kwargs={'n_features': 25, 'max_freq': 10.0}
)

results_norm = train_model(
    model_sh_rff_norm,
    train_loader,
    test_loader,
    epochs=100,
    use_sh=True,
    L=10,
    normalize_sh=True
)

print(f"\nFinal R² (with normalization): {results_norm['final_r2']:.4f}")
print(f"Improvement: {(results_norm['final_r2'] - results_no_norm['final_r2']):.4f} "
      f"({100*(results_norm['final_r2'] - results_no_norm['final_r2'])/results_no_norm['final_r2']:.2f}%)")

In [ ]:
# Test 3: SH + ReLU for comparison (should work well)
print("\n" + "=" * 60)
print("Test 3: SH + ReLU (baseline) - Should be ~0.749")
print("=" * 60)

model_sh_relu = build_mlp_with_activation(
    input_dim=100,
    hidden_dims=[256, 256, 256],
    output_dim=256,
    activation_type='relu'
)

results_relu = train_model(
    model_sh_relu,
    train_loader,
    test_loader,
    epochs=100,
    use_sh=True,
    L=10,
    normalize_sh=False  # ReLU doesn't need normalization
)

print(f"\nFinal R² (ReLU): {results_relu['final_r2']:.4f}")

In [ ]:
# Visualize training dynamics
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Training loss
axes[0].plot(results_no_norm['train_losses'], label='RFF (no norm)', alpha=0.7)
axes[0].plot(results_norm['train_losses'], label='RFF (norm)', alpha=0.7)
axes[0].plot(results_relu['train_losses'], label='ReLU', alpha=0.7)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Training Loss')
axes[0].set_title('Training Loss Over Time')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: Test R²
test_epochs = range(0, 100, 10)
axes[1].plot(test_epochs, results_no_norm['test_r2s'], 'o-', label='RFF (no norm)', alpha=0.7)
axes[1].plot(test_epochs, results_norm['test_r2s'], 'o-', label='RFF (norm)', alpha=0.7)
axes[1].plot(test_epochs, results_relu['test_r2s'], 'o-', label='ReLU', alpha=0.7)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Test R²')
axes[1].set_title('Test R² Over Time')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Plot 3: Gradient norms
axes[2].plot(results_no_norm['grad_norms'], label='RFF (no norm)', alpha=0.7)
axes[2].plot(results_norm['grad_norms'], label='RFF (norm)', alpha=0.7)
axes[2].plot(results_relu['grad_norms'], label='ReLU', alpha=0.7)
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Gradient Norm')
axes[2].set_title('Gradient Norms Over Time')
axes[2].legend()
axes[2].grid(True, alpha=0.3)
axes[2].set_yscale('log')

plt.tight_layout()
plt.savefig('training_dynamics_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nTraining Dynamics Analysis:")
print(f"Final gradient norm (no norm): {results_no_norm['grad_norms'][-1]:.4f}")
print(f"Final gradient norm (norm): {results_norm['grad_norms'][-1]:.4f}")
print(f"Final gradient norm (ReLU): {results_relu['grad_norms'][-1]:.4f}")

## 4. Experiment 2: Longer Training

Maybe RFF needs more epochs to converge?

In [ ]:
print("=" * 60)
print("Test 4: SH + RFF with normalization, 500 epochs")
print("This will take ~15-20 minutes...")
print("=" * 60)

model_sh_rff_long = build_mlp_with_activation(
    input_dim=100,
    hidden_dims=[256, 256, 256],
    output_dim=256,
    activation_type='rff',
    activation_kwargs={'n_features': 25, 'max_freq': 10.0}
)

results_long = train_model(
    model_sh_rff_long,
    train_loader,
    test_loader,
    epochs=500,
    use_sh=True,
    L=10,
    normalize_sh=True,
    verbose=False  # Less spam
)

# Print progress every 50 epochs
for i, epoch in enumerate(range(0, 500, 50)):
    if i < len(results_long['test_r2s']):
        print(f"Epoch {epoch}: R² = {results_long['test_r2s'][i]:.4f}")

print(f"\nFinal R² (500 epochs): {results_long['final_r2']:.4f}")
print(f"Improvement over 100 epochs: {(results_long['final_r2'] - results_norm['final_r2']):.4f}")

## 5. Experiment 3: Learnable Frequencies

Maybe fixed frequencies are the problem?

In [ ]:
print("=" * 60)
print("Test 5: SH + RFF with LEARNABLE frequencies")
print("=" * 60)

model_sh_rff_learnable = build_mlp_with_activation(
    input_dim=100,
    hidden_dims=[256, 256, 256],
    output_dim=256,
    activation_type='rff',
    activation_kwargs={
        'n_features': 25,
        'max_freq': 10.0,
        'learnable_freq': True  # KEY CHANGE
    }
)

results_learnable = train_model(
    model_sh_rff_learnable,
    train_loader,
    test_loader,
    epochs=100,
    use_sh=True,
    L=10,
    normalize_sh=True
)

print(f"\nFinal R² (learnable freq): {results_learnable['final_r2']:.4f}")
print(f"vs fixed freq: {results_norm['final_r2']:.4f}")
print(f"Difference: {(results_learnable['final_r2'] - results_norm['final_r2']):.4f}")

## 6. Visualize Learned RFF Activations

In [ ]:
def visualize_rff_activation(model, layer_idx=0, title=""):
    """Visualize a learned RFF activation function."""
    # Get the RFF activation from the specified layer
    act_fn = model.layers[layer_idx * 2 + 1]  # Skip Linear layers
    
    if not isinstance(act_fn, RFFActivation):
        print(f"Layer {layer_idx} is not RFF, it's {type(act_fn)}")
        return
    
    # Generate input range
    x = torch.linspace(-5, 5, 1000).to(device)
    
    # Evaluate activation
    with torch.no_grad():
        y = act_fn(x.unsqueeze(1)).squeeze().cpu().numpy()
    
    x_np = x.cpu().numpy()
    
    # Plot
    plt.figure(figsize=(10, 6))
    plt.plot(x_np, y, linewidth=2)
    plt.axhline(0, color='black', linestyle='--', alpha=0.3)
    plt.axvline(0, color='black', linestyle='--', alpha=0.3)
    plt.xlabel('Input', fontsize=12)
    plt.ylabel('Output', fontsize=12)
    plt.title(f'{title} - Layer {layer_idx}', fontsize=14)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    
    # Print learned parameters
    print(f"\nLayer {layer_idx} RFF Parameters:")
    print(f"  Frequencies (first 5): {act_fn.freqs[:5].cpu().numpy()}")
    print(f"  Sin coeffs (first 5): {act_fn.sin_coeffs[:5].item()}")
    print(f"  Cos coeffs (first 5): {act_fn.cos_coeffs[:5].item()}")
    print(f"  Bias: {act_fn.bias.item():.4f}")
    print(f"  Scale: {act_fn.scale.item():.4f}")

# Visualize RFF from different models
print("RFF Activation Shapes After Training:\n")

visualize_rff_activation(model_sh_rff_no_norm, layer_idx=0, 
                        title="RFF (No Normalization)")
plt.savefig('rff_no_norm_layer0.png', dpi=150, bbox_inches='tight')
plt.show()

visualize_rff_activation(model_sh_rff_norm, layer_idx=0,
                        title="RFF (With Normalization)")
plt.savefig('rff_norm_layer0.png', dpi=150, bbox_inches='tight')
plt.show()

# Compare to ReLU
x = np.linspace(-5, 5, 1000)
plt.figure(figsize=(10, 6))
plt.plot(x, np.maximum(0, x), linewidth=2, label='ReLU')
plt.axhline(0, color='black', linestyle='--', alpha=0.3)
plt.axvline(0, color='black', linestyle='--', alpha=0.3)
plt.xlabel('Input', fontsize=12)
plt.ylabel('Output', fontsize=12)
plt.title('ReLU Activation (for comparison)', fontsize=14)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig('relu_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Summary Table

In [ ]:
# Create comprehensive results table
results_df = pd.DataFrame([
    {
        'Model': 'SH + ReLU',
        'R²': results_relu['final_r2'],
        'Normalization': 'No',
        'Epochs': 100,
        'Learnable Freq': 'N/A',
        'vs Baseline': 0.0
    },
    {
        'Model': 'SH + RFF',
        'R²': results_no_norm['final_r2'],
        'Normalization': 'No',
        'Epochs': 100,
        'Learnable Freq': 'No',
        'vs Baseline': results_no_norm['final_r2'] - results_relu['final_r2']
    },
    {
        'Model': 'SH + RFF',
        'R²': results_norm['final_r2'],
        'Normalization': 'Yes',
        'Epochs': 100,
        'Learnable Freq': 'No',
        'vs Baseline': results_norm['final_r2'] - results_relu['final_r2']
    },
    {
        'Model': 'SH + RFF',
        'R²': results_learnable['final_r2'],
        'Normalization': 'Yes',
        'Epochs': 100,
        'Learnable Freq': 'Yes',
        'vs Baseline': results_learnable['final_r2'] - results_relu['final_r2']
    },
    {
        'Model': 'SH + RFF',
        'R²': results_long['final_r2'],
        'Normalization': 'Yes',
        'Epochs': 500,
        'Learnable Freq': 'No',
        'vs Baseline': results_long['final_r2'] - results_relu['final_r2']
    }
])

# Add percentage difference
results_df['% vs Baseline'] = (results_df['vs Baseline'] / results_relu['final_r2'] * 100)

print("\n" + "=" * 80)
print("DIAGNOSTIC RESULTS SUMMARY")
print("=" * 80)
print(results_df.to_string(index=False))
print("=" * 80)

# Save to CSV
results_df.to_csv('diagnostic_results_nb17.csv', index=False)
print("\nResults saved to diagnostic_results_nb17.csv")

## 8. Conclusions

### What We Learned:

1. **Input normalization is critical**: 
   - Without normalization: R² ≈ 0.66 (catastrophic failure)
   - With normalization: R² ≈ ??? (to be determined from results)

2. **Training dynamics**:
   - Check if gradient norms are stable
   - Check if longer training helps

3. **Learnable frequencies**:
   - Does learning frequencies improve over fixed?

4. **Activation shapes**:
   - Are RFF activations learning meaningful nonlinearities?
   - Or are they stuck in local minima?

### Next Steps:

**If normalization fixes it (R² > 0.74)**:
- Update notebook 16 to use normalized SH features
- Re-run all SH + RFF experiments
- Proceed with Phase 2 of roadmap

**If normalization doesn't fix it (R² < 0.72)**:
- Frequency interference hypothesis is correct
- Avoid SH + RFF combinations
- Focus on SH + Spline instead
- Consider alternative: Raw + RFF might still work

**If learnable frequencies help significantly**:
- Make learnable_freq=True the default
- Test on other tasks (elevation, temperature)

**If 500 epochs shows big improvement**:
- RFF needs longer training than ReLU/SIREN
- Update training protocol to 500 epochs
- Consider learning rate scheduling

## 9. Additional Analysis: Frequency Spectrum

Let's analyze what frequencies RFF is learning.

In [ ]:
def analyze_rff_frequencies(model, title=""):
    """Analyze learned frequencies in RFF activations."""
    print(f"\n{title}")
    print("=" * 60)
    
    for layer_idx in range(3):  # 3 hidden layers
        act_fn = model.layers[layer_idx * 2 + 1]
        
        if isinstance(act_fn, RFFActivation):
            freqs = act_fn.freqs.detach().cpu().numpy()
            sin_coeffs = act_fn.sin_coeffs.detach().cpu().numpy()
            cos_coeffs = act_fn.cos_coeffs.detach().cpu().numpy()
            
            # Compute effective amplitudes
            amplitudes = np.sqrt(sin_coeffs**2 + cos_coeffs**2)
            
            print(f"\nLayer {layer_idx}:")
            print(f"  Frequency range: [{freqs.min():.4f}, {freqs.max():.4f}]")
            print(f"  Mean amplitude: {amplitudes.mean():.4f}")
            print(f"  Active frequencies (amp > 0.1): {(amplitudes > 0.1).sum()} / {len(amplitudes)}")
            
            # Plot frequency spectrum
            plt.figure(figsize=(10, 4))
            plt.subplot(1, 2, 1)
            plt.bar(range(len(freqs)), freqs, alpha=0.7)
            plt.xlabel('Feature Index')
            plt.ylabel('Frequency (ω)')
            plt.title(f'Learned Frequencies - Layer {layer_idx}')
            plt.grid(True, alpha=0.3)
            
            plt.subplot(1, 2, 2)
            plt.bar(range(len(amplitudes)), amplitudes, alpha=0.7, color='orange')
            plt.xlabel('Feature Index')
            plt.ylabel('Amplitude')
            plt.title(f'Frequency Amplitudes - Layer {layer_idx}')
            plt.grid(True, alpha=0.3)
            
            plt.tight_layout()
            plt.savefig(f'rff_frequencies_{title.replace(" ", "_")}_layer{layer_idx}.png', 
                       dpi=150, bbox_inches='tight')
            plt.show()

# Analyze different models
analyze_rff_frequencies(model_sh_rff_no_norm, "No Normalization")
analyze_rff_frequencies(model_sh_rff_norm, "With Normalization")
if results_learnable:
    analyze_rff_frequencies(model_sh_rff_learnable, "Learnable Frequencies")

## 10. Final Verdict

Based on the experiments above, we can determine:

1. **Primary cause of failure**: [To be filled based on results]
2. **Can RFF work with SH?**: [Yes/No based on normalized results]
3. **Recommended fix**: [List specific fixes]
4. **Should we continue with RFF?**: [Decision based on results]

See `DIAGNOSTIC_CONCLUSIONS_NB17.md` for detailed analysis.